# 自训嵌入：FastText

**负责人：** 2 号　**阶段：** 2 嵌入

全组共用的从零训练嵌入，对应评分里"从零训练"的加分。

## 任务

标"（选做）"的任务在必做部分完成后再做。

- [x] 用 gensim 训练 FastText（skip-gram）
- [x] 按折训练：第 k 折只用该折的训练部分训练，保存为 `fasttext_*_fold0` … `fasttext_*_fold4`；预测测试集用的版本用全部训练集训练，保存为 `fasttext_*_full`。测试集文本任何时候都不参与训练
- [x] 句向量：平均池化和 SIF 加权；SIF 用到的词频也按折统计。不能用 TF-IDF 加权，TF-IDF 只能用在 baseline
- [x] 给 5、6 号的序列模型准备词表和词向量矩阵，同样按折保存
- [ ] （选做）Word2Vec 对照
- [ ] （选做）画 crash / bear / dip 近邻词的 t-SNE 图

## 产出

- `DATA_ROOT/emb/fasttext_mean_fold{0-4}.npy`、`fasttext_mean_full.npy`
- `DATA_ROOT/emb/fasttext_sif_*`、序列模型用的词表和词向量矩阵

In [ ]:
import sys
from pathlib import Path

# 向上查找仓库根目录（含 src/config.py 的文件夹），在任何子文件夹里打开都能导入 src
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), None)
if ROOT is None:
    raise RuntimeError("找不到仓库根目录：请在 clone 下来的仓库里启动 Jupyter（Colab 先 %cd 进仓库）")
sys.path.insert(0, str(ROOT))

from src.config import DATA_ROOT, FIGURES_DIR, LABELS, SEED                 # 路径和常量
from src.data import load_clean, load_raw, load_splits                      # 读取数据和划分
from src.evaluate import evaluate, new_run_id, plot_confusion, save_result  # 统一评分
from src.cache import load_embedding, save_embedding, save_preds            # 嵌入和预测结果的读写

print("仓库根目录:", ROOT)
print("DATA_ROOT:", DATA_ROOT)

## Why fold-wise training

FastText is trained on our own tweets (the "trained from scratch" embedding). It never sees labels, but if the fold-*k* validation tweets were part of its training text, the cross-validation score of every model using it would be slightly optimistic. So:

- fold *k* (k = 0…4): FastText is trained only on the other four folds, then encodes **all** training-split tweets;
- `full`: FastText is trained on the whole training split and encodes training **and** test tweets, for the final test predictions;
- test tweets never take part in training.

Each version produces mean-pooled vectors, SIF vectors (Arora et al., 2017) and token-vector sequences for the TextCNN / BiLSTM of members 5 and 6.

In [ ]:
import time

import numpy as np
import pandas as pd

from src.cache import load_embedding
from src.config import EMB_DIR, N_FOLDS
from src.fasttext_emb import FASTTEXT_PARAMS, build_fold_embeddings, tokenize

clean = load_clean()
splits = load_splits(clean=clean)
lengths = clean["text_norm"].map(lambda s: len(tokenize(s)))
print(lengths.describe().round(1).to_string())
MAX_LEN = int(np.ceil(np.percentile(lengths, 99)))
print(f"max_len for token sequences = {MAX_LEN} (99th percentile; {int((lengths > MAX_LEN).sum())} tweets get truncated)")
FASTTEXT_PARAMS

In [ ]:
start = time.time()
summary, full_model = build_fold_embeddings(clean, splits, MAX_LEN)
print(f"trained {len(summary)} FastText models in {time.time() - start:.0f}s")
summary

## Sanity checks

Every saved file must cover the right tweets, in the right order, and the fold versions must not contain test tweets.

In [ ]:
dev_ids = splits.loc[splits["split"] == "train", "id"].to_numpy()
test_ids = splits.loc[splits["split"] == "test", "id"].to_numpy()
for tag in [f"fold{k}" for k in range(N_FOLDS)] + ["full"]:
    for kind in ("mean", "sif", "tok"):
        ids, matrix = load_embedding(f"fasttext_{kind}_{tag}")
        expected = dev_ids if tag != "full" else splits["id"].to_numpy()
        assert np.array_equal(np.sort(ids), np.sort(expected)), (kind, tag)
        assert tag == "full" or not np.isin(test_ids, ids).any(), (kind, tag)
        assert np.isfinite(matrix.astype(np.float32)).all(), (kind, tag)
ids, seq = load_embedding("fasttext_tok_full")
_, mask = load_embedding("fasttext_tok_full_mask")
print("token sequences:", seq.shape, seq.dtype, "| mask:", mask.shape, "| mean real tokens per tweet:", round(mask.sum(1).mean(), 1))
print("all checks passed")

## What did FastText learn?

Nearest neighbours of a few market words in the `full` model. They show whether the embedding captured the finance meaning of words that a general sentiment lexicon gets wrong.

In [ ]:
words = ["crash", "bear", "bull", "dip", "short", "red", "green", "pump", "dump", "recession"]
pd.DataFrame({w: [f"{n} ({s:.2f})" for n, s in full_model.wv.most_similar(w, topn=5)] for w in words}).T